In [1]:
import pandas as pd
from pathlib import Path
import re
from tqdm import tqdm
import gemmi
import numpy as np
from cctbx import crystal
from cctbx import sgtbx
from collections import defaultdict
import copy

from mlindex.optimization.CandidateValidation import validate_candidate_known_bl
from mlindex.utilities.UnitCellTools import get_partial_unit_cell
from mlindex.utilities.Reindexing import selling_reduction
from mlindex.utilities.Reindexing import rhombohedral_to_hexagonal
%load_ext line_profiler

In [2]:
base_dir = '/global/cfs/cdirs/m4064/dwmoreau/MLI/mlindex/comparisons/'

In [66]:
file_name = '/global/cfs/cdirs/m4064/dwmoreau/MLI/mlindex/data/opxrd/CNRS_output_data_verified_final3.json'
df = pd.read_json(file_name)

In [14]:
#df = df[df.bravais_lattice == 'cP']

In [15]:
dicvol_file_name = Path(base_dir, 'results', 'results_dicvol.h5')
with pd.HDFStore(dicvol_file_name) as store:
    results_dicvol = {key: store[key] for key in store.keys()}

In [16]:
ito_file_name = Path(base_dir, 'results', 'results_ito.h5')
with pd.HDFStore(ito_file_name) as store:
    results_ito = {key: store[key] for key in store.keys()}

In [17]:
treor_file_name = Path(base_dir, 'results', 'results_treor.h5')
with pd.HDFStore(treor_file_name) as store:
    results_treor = {key: store[key] for key in store.keys()}

In [18]:
gsas_file_name = Path(base_dir, 'results', 'results_gsas.h5')
with pd.HDFStore(gsas_file_name) as store:
    results_gsas = {key: store[key] for key in store.keys()}

In [67]:
FOM_THRESHOLD = 100

LATTICE_SYSTEM_TO_SPACE_GROUP = {
    'cubic': 'P23',
    'tetragonal': 'P4',
    'hexagonal': 'P6',
    'rhombohedral': 'P6',
    'orthorhombic': 'P222',
    'monoclinic': 'P2',
    'triclinic': 'P1',
}

def niggli_reduction(unit_cell, spacegroup):
    sym = crystal.symmetry(
        unit_cell=unit_cell,
        space_group_symbol=spacegroup,
        correct_rhombohedral_setting_if_necessary=True,
    )
    return sym.best_cell().unit_cell().parameters()


def check_niggli(unit_cell, space_group, true_niggli_unit_cell):
    """unit_cell in degrees. Returns True if Niggli-reduced cell matches."""
    try:
        niggli_unit_cell = niggli_reduction(list(unit_cell), space_group)
        return np.all(np.isclose(true_niggli_unit_cell, niggli_unit_cell, rtol=0.02))
    except Exception:
        return False


def check_s6(unit_cell_rad, true_s6_reduced):
    """unit_cell_rad in radians. Returns True if S6-reduced cell matches."""
    _, _, s6_reduced = selling_reduction(unit_cell_rad[np.newaxis])
    return np.all(np.isclose(true_s6_reduced, s6_reduced[0], rtol=0.02))


def check_partial_bl(unit_cell_rad, lattice_system, true_entry):
    """unit_cell_rad in radians. Returns True if partial unit cell matches."""
    if lattice_system != true_entry.lattice_system:
        return False
    unit_cell_partial = get_partial_unit_cell(unit_cell_rad, lattice_system=lattice_system)
    found, _ = validate_candidate_known_bl(
        np.array(true_entry.unit_cell),
        unit_cell_partial,
        true_entry.bravais_lattice,
        rtol=0.02
    )
    return found


def run_strategies(unit_cell_deg, unit_cell_rad, lattice_system, space_group,
                   true_entry, true_niggli_unit_cell, true_s6_reduced):
    """Run all three validation strategies. Returns True if any match."""
    if check_niggli(unit_cell_deg, space_group, true_niggli_unit_cell):
        return True
    if check_s6(unit_cell_rad, true_s6_reduced):
        return True
    if check_partial_bl(unit_cell_rad, lattice_system, true_entry):
        return True
    return False


def unit_cell_to_bl(unit_cell, centering):
    angles = list(unit_cell[3:])
    if angles == [90, 90, 90]:
        uc_sorted = np.sort(unit_cell[:3])
        if uc_sorted[0] == uc_sorted[1]:
            if uc_sorted[1] == uc_sorted[2]:
                return {
                    'P': ('cP', 'cubic'),
                    'I': ('cI', 'cubic'),
                    'F': ('cF', 'cubic')
                }.get(centering, ('cP', 'cubic'))
            else:
                return {
                    'P': ('tP', 'tetragonal'),
                    'I': ('tI', 'tetragonal'),
                    'F': ('tF', 'tetragonal')
                }.get(centering, ('tP', 'tetragonal'))
        else:
            return {
                'P': ('oP', 'orthorhombic'),
                'I': ('oI', 'orthorhombic'),
                'F': ('oF', 'orthorhombic'),
                'A': ('oA', 'orthorhombic'),
                'C': ('oC', 'orthorhombic')
            }.get(centering, ('oP', 'orthorhombic'))
    elif [unit_cell[3], unit_cell[5]] == [90, 90]:
        return {
            'P': ('mP', 'monoclinic'),
            'I': ('mI', 'monoclinic'),
            'A': ('mA', 'monoclinic'),
            'B': ('mB', 'monoclinic'),
            'C': ('mC', 'monoclinic'),
            'F': ('mF', 'monoclinic')
        }.get(centering, ('mP', 'monoclinic'))
    elif unit_cell[5] == 120:
        return {
            'P': ('hP', 'hexagonal'),
            'R': ('hR', 'rhombohedral')
        }.get(centering, (None, None))
    else:
        return 'aP', 'triclinic'


def validate_gsas(df, true_entry, true_niggli_unit_cell, true_s6_reduced):
    for index in range(len(df)):
        entry = df.iloc[index]
        unit_cell_deg = [entry.a, entry.b, entry.c, entry.alpha, entry.beta, entry.gamma]
        unit_cell_deg = np.array([90.0 if np.isnan(i) else float(i) for i in unit_cell_deg])
        unit_cell_rad = unit_cell_deg.copy()
        unit_cell_rad[3:] *= np.pi / 180
        lattice_system = entry.lattice_system
        if lattice_system == 'rhombohedral':
            lattice_system = 'hexagonal'
        space_group = entry.spacegroup
        if run_strategies(unit_cell_deg, unit_cell_rad, lattice_system, space_group,
                          true_entry, true_niggli_unit_cell, true_s6_reduced):
            return True
    if df['M20'].max() > FOM_THRESHOLD:
        return True
    return False


def validate_treor(df, true_entry, true_niggli_unit_cell, true_s6_reduced):
    for index in range(len(df)):
        entry = df.iloc[index]
        unit_cell_deg = [entry.A, entry.B, entry.C, entry.Alpha, entry.Beta, entry.Gamma]
        unit_cell_deg = np.array([90.0 if np.isnan(i) else float(i) for i in unit_cell_deg])
        unit_cell_rad = unit_cell_deg.copy()
        unit_cell_rad[3:] *= np.pi / 180
        lattice_system = entry.lattice_system.lower()
        if lattice_system == 'rhombohedral':
            lattice_system = 'hexagonal'
        space_group = LATTICE_SYSTEM_TO_SPACE_GROUP[lattice_system]
        if run_strategies(unit_cell_deg, unit_cell_rad, lattice_system, space_group,
                          true_entry, true_niggli_unit_cell, true_s6_reduced):
            return True
    if df['M'].max() > FOM_THRESHOLD:
        return True
    return False


def validate_dicvol(df, true_entry, true_niggli_unit_cell, true_s6_reduced):
    for index in range(len(df)):
        entry = df.iloc[index]
        unit_cell_deg = [entry.A, entry.B, entry.C, entry.Alpha, entry.Beta, entry.Gamma]
        unit_cell_deg = np.array([90.0 if np.isnan(i) else float(i) for i in unit_cell_deg])
        unit_cell_rad = unit_cell_deg.copy()
        unit_cell_rad[3:] *= np.pi / 180
        lattice_system = entry.System.lower()
        if lattice_system == 'rhombohedral':
            lattice_system = 'hexagonal'
        space_group = LATTICE_SYSTEM_TO_SPACE_GROUP[lattice_system]
        if run_strategies(unit_cell_deg, unit_cell_rad, lattice_system, space_group,
                          true_entry, true_niggli_unit_cell, true_s6_reduced):
            return True
    if df['M'].max() > FOM_THRESHOLD:
        return True
    return False


def validate_ito(df, true_entry, true_niggli_unit_cell, true_s6_reduced):
    for index in range(len(df)):
        entry = df.iloc[index]
        unit_cell_deg = np.array(entry['Unit Cell'], dtype=float)
        unit_cell_rad = unit_cell_deg.copy()
        unit_cell_rad[3:] *= np.pi / 180
        bl, lattice_system = unit_cell_to_bl(unit_cell_deg, entry['Centering'])
        if lattice_system == 'rhombohedral':
            lattice_system = 'hexagonal'
        space_group = LATTICE_SYSTEM_TO_SPACE_GROUP[lattice_system]

        if run_strategies(unit_cell_deg, unit_cell_rad, lattice_system, space_group,
                          true_entry, true_niggli_unit_cell, true_s6_reduced):
            return True

        # ITO-specific: attempt best_cell correction when lattice systems don't match
        if lattice_system != true_entry.lattice_system and lattice_system != 'triclinic':
            if lattice_system == 'tetragonal' and unit_cell_deg[0] != unit_cell_deg[1]:
                if unit_cell_deg[0] == unit_cell_deg[2]:
                    unit_cell_deg = np.array([unit_cell_deg[0], unit_cell_deg[2], unit_cell_deg[1], 90, 90, 90])
                elif unit_cell_deg[1] == unit_cell_deg[2]:
                    unit_cell_deg = np.array([unit_cell_deg[1], unit_cell_deg[2], unit_cell_deg[0], 90, 90, 90])
            try:
                sym = crystal.symmetry(unit_cell=unit_cell_deg.tolist(), space_group_symbol=space_group)
                unit_cell_deg = np.array(sym.best_cell().unit_cell().parameters())
                unit_cell_rad = unit_cell_deg.copy()
                unit_cell_rad[3:] *= np.pi / 180
                bl, lattice_system = unit_cell_to_bl(unit_cell_deg, 'P')
                space_group = LATTICE_SYSTEM_TO_SPACE_GROUP[lattice_system]
                if check_partial_bl(unit_cell_rad, lattice_system, true_entry):
                    return True
            except Exception:
                print(unit_cell_deg.tolist(), lattice_system, bl, space_group)

    if df['Figure of Merit'].max() > FOM_THRESHOLD:
        return True
    return False


def validate_mlindex(df, true_entry, true_niggli_unit_cell, true_s6_reduced):
    df = df[df['M20'] > 10]
    if len(df) > 10:
        #cubic_lattices = np.stack((
        #    df.bravais_lattice == 'cP',
        #    df.bravais_lattice == 'cI',
        #    df.bravais_lattice == 'cF',
        #), axis=1).any(axis=1)
        #n_peaks = 20*np.ones(len(df))
        #n_peaks[cubic_lattices] = 10
        #X20 = 1 + (n_peaks - df['n_indexed'])
        #df['FOM'] = df['M20'] / X20
        #df = df.sort_values('FOM', ascending=False).head(10)
        df = df.head(10)
        
    for index in range(len(df)):
        entry = df.iloc[index]
        unit_cell_rad = np.array([entry.a, entry.b, entry.c, entry.alpha, entry.beta, entry.gamma], dtype=float)
        unit_cell_deg = unit_cell_rad.copy()
        unit_cell_deg[3:] *= 180 / np.pi

        if entry.bravais_lattice == 'hP':
            unit_cell_deg = np.array([unit_cell_deg[0], unit_cell_deg[1], unit_cell_deg[2], 90, 90, 120])

        space_group = entry.spacegroup.split('e.g.')[1].strip()

        if check_s6(unit_cell_rad, true_s6_reduced):
            return True
        if check_niggli(unit_cell_deg, space_group, true_niggli_unit_cell):
            return True
        if check_partial_bl(unit_cell_rad, true_entry.lattice_system, true_entry):
            return True

    return False

In [68]:
success_ito = defaultdict(int)
success_dicvol = defaultdict(int)
success_treor = defaultdict(int)
success_gsas = defaultdict(int)
success_mlindex = defaultdict(int)
success_any = defaultdict(int)
counts = defaultdict(int)
for index in tqdm(range(len(df))):
    entry = df.iloc[index]
    bravais_lattice = entry.bravais_lattice
    counts[bravais_lattice] += 1
    pattern = Path(entry.file_name).name.split('.')[0]
    unit_cell_radians = np.array(entry.unit_cell)
    unit_cell = unit_cell_radians.copy()
    unit_cell[3:] *= 180/np.pi
    niggli_unit_cell = niggli_reduction(
        unit_cell.tolist(),
        entry.spacegroup
    )
    _, _, s6_reduced = selling_reduction(unit_cell_radians[np.newaxis])
    s6_reduced = s6_reduced[0]

    found_any = False

    # Do mlindex first since it has the rhombohedral setting
    opt_results = pd.read_json(entry.file_name.replace('.json', '_opt.json'))
    #print(pattern)
    #print(entry)
    #print(opt_results)
    if validate_mlindex(opt_results, entry, niggli_unit_cell, s6_reduced):
        success_mlindex[bravais_lattice] += 1
        found_any = True

    if entry.lattice_system == 'rhombohedral':
        entry = copy.deepcopy(entry)
        entry.lattice_system = 'hexagonal'
        entry.bravais_lattice = 'hP'
        entry.unit_cell = rhombohedral_to_hexagonal(entry.unit_cell)
        unit_cell_radians = entry.unit_cell
        unit_cell = unit_cell_radians.copy()
        unit_cell[3:] *= 180/np.pi
        niggli_unit_cell = niggli_reduction(
            unit_cell.tolist(),
            entry.spacegroup
        )
        _, _, s6_reduced = selling_reduction(unit_cell_radians[np.newaxis])
        s6_reduced = s6_reduced[0]

    if validate_ito(results_ito[f'/{pattern}'], entry, niggli_unit_cell, s6_reduced):
        success_ito[bravais_lattice] += 1
        found_any = True
    if f'/{pattern}' in results_dicvol.keys():
        if validate_dicvol(results_dicvol[f'/{pattern}'], entry, niggli_unit_cell, s6_reduced):
            success_dicvol[bravais_lattice] += 1
            found_any = True
    if f'/{pattern}' in results_treor.keys():
        if validate_treor(results_treor[f'/{pattern}'], entry, niggli_unit_cell, s6_reduced):
            success_treor[bravais_lattice] += 1
            found_any = True
    if f'/{pattern}' in results_gsas.keys():
        #print(results_gsas[f'/{pattern}'])
        if validate_gsas(results_gsas[f'/{pattern}'], entry, niggli_unit_cell, s6_reduced):
            #print('SUCCESS')
            success_gsas[bravais_lattice] += 1
            found_any = True
    #print()
    #print()
    
    if found_any:
        success_any[bravais_lattice] += 1

/global/cfs/cdirs/m4064/dwmoreau/MLI/mlindex/optimization/CandidateValidation.py:105: RuntimeWarning: invalid value encountered in sqrt
  cz = unit_cell_pred[0] * np.sqrt(np.sin(unit_cell_pred[1])**2 - arg**2)
/global/cfs/cdirs/m4064/dwmoreau/MLI/mlindex/optimization/CandidateValidation.py:105: RuntimeWarning: invalid value encountered in sqrt
  cz = unit_cell_pred[0] * np.sqrt(np.sin(unit_cell_pred[1])**2 - arg**2)
/global/cfs/cdirs/m4064/dwmoreau/MLI/mlindex/optimization/CandidateValidation.py:105: RuntimeWarning: invalid value encountered in sqrt
  cz = unit_cell_pred[0] * np.sqrt(np.sin(unit_cell_pred[1])**2 - arg**2)
/global/cfs/cdirs/m4064/dwmoreau/MLI/mlindex/optimization/CandidateValidation.py:105: RuntimeWarning: invalid value encountered in sqrt
  cz = unit_cell_pred[0] * np.sqrt(np.sin(unit_cell_pred[1])**2 - arg**2)
100%|██████████| 599/599 [04:59<00:00,  2.00it/s]


In [69]:
for bl in ['cP', 'cI', 'cF', 'tP', 'tI', 'hP', 'hR', 'oP', 'oC', 'oF', 'oI', 'mP', 'mC', 'aP']:
    print(' | '.join((
        f'   {bl}',
        f'{success_dicvol[bl]:3d}',
        f'{success_ito[bl]:3d}',
        f'{success_treor[bl]:3d}',
        f'{success_gsas[bl]:3d}',
        f'{success_mlindex[bl]:3d}',
        f'{success_any[bl]:3d}',
        f'{counts[bl]:3d}',
    )))
print(' | '.join((
    f'Total',
    f'{sum(success_dicvol.values()):3d}',
    f'{sum(success_ito.values()):3d}',
    f'{sum(success_treor.values()):3d}',
    f'{sum(success_gsas.values()):3d}',
    f'{sum(success_mlindex.values()):3d}',
    f'{sum(success_any.values()):3d}',
    f'{sum(counts.values()):3d}'
)))


   cP |   9 |   5 |  11 |  11 |  17 |  18 |  21
   cI |   3 |   2 |   3 |   3 |   2 |   3 |   4
   cF |   6 |   6 |   8 |   8 |  17 |  18 |  22
   tP |  38 |  27 |  39 |  19 |  46 |  50 |  56
   tI |  10 |   9 |   6 |  14 |  19 |  22 |  30
   hP |  15 |   5 |  23 |  20 |  32 |  33 |  36
   hR |   2 |   4 |   3 |   8 |  11 |  14 |  19
   oP |  63 |  63 |  77 |  77 | 101 | 116 | 124
   oC |   2 |   9 |   5 |  10 |  10 |  12 |  13
   oF |   1 |   1 |   1 |   2 |   2 |   2 |   2
   oI |   5 |   3 |   4 |   9 |   9 |  10 |  16
   mP |  72 |  51 |  48 |  96 | 108 | 125 | 149
   mC |  20 |  16 |  11 |  37 |  37 |  43 |  51
   aP |  14 |   2 |  23 |  25 |  39 |  43 |  56
Total | 260 | 203 | 262 | 339 | 450 | 509 | 599


In [70]:
bls = ['cP', 'cI', 'cF', 'tP', 'tI', 'hP', 'hR', 'oP', 'oC', 'oF', 'oI', 'mP', 'mC', 'aP']
dicts = [success_dicvol, success_ito, success_treor, success_gsas, success_mlindex, success_any, counts]
headers = ['DICVOL', 'ITO', 'TREOR', 'GSAS', 'MLindex', 'Any', 'Total']

col_fmt = 'l' + 'r' * len(headers)
header_row = 'BL & ' + ' & '.join(headers) + r' \\'

rows = []
for bl in bls:
    values = ' & '.join(f'{d[bl]:3d}' for d in dicts)
    rows.append(f'    {bl} & {values} ' + r'\\')

totals = ' & '.join(f'{sum(d.values()):3d}' for d in dicts)
total_row = f'    Total & {totals} ' + r'\\'

print(r'\begin{table}[h]')
print(r'    \centering')
print(r'    \begin{tabular}{' + col_fmt + '}')
print(r'    \toprule')
print(f'    {header_row}')
print(r'    \midrule')
print('\n'.join(rows))
print(r'    \midrule')
print(f'    {total_row}')
print(r'    \bottomrule')
print(r'    \end{tabular}')
print(r'    \caption{Your caption here.}')
print(r'    \label{tab:your_label}')
print(r'\end{table}')

\begin{table}[h]
    \centering
    \begin{tabular}{lrrrrrrr}
    \toprule
    BL & DICVOL & ITO & TREOR & GSAS & MLindex & Any & Total \\
    \midrule
    cP &   9 &   5 &  11 &  11 &  17 &  18 &  21 \\
    cI &   3 &   2 &   3 &   3 &   2 &   3 &   4 \\
    cF &   6 &   6 &   8 &   8 &  17 &  18 &  22 \\
    tP &  38 &  27 &  39 &  19 &  46 &  50 &  56 \\
    tI &  10 &   9 &   6 &  14 &  19 &  22 &  30 \\
    hP &  15 &   5 &  23 &  20 &  32 &  33 &  36 \\
    hR &   2 &   4 &   3 &   8 &  11 &  14 &  19 \\
    oP &  63 &  63 &  77 &  77 & 101 & 116 & 124 \\
    oC &   2 &   9 &   5 &  10 &  10 &  12 &  13 \\
    oF &   1 &   1 &   1 &   2 &   2 &   2 &   2 \\
    oI &   5 &   3 &   4 &   9 &   9 &  10 &  16 \\
    mP &  72 &  51 &  48 &  96 & 108 & 125 & 149 \\
    mC &  20 &  16 &  11 &  37 &  37 &  43 &  51 \\
    aP &  14 &   2 &  23 &  25 &  39 &  43 &  56 \\
    \midrule
        Total & 260 & 203 & 262 & 339 & 450 & 509 & 599 \\
    \bottomrule
    \end{tabular}
    \caption{Y